# PDF to Markdown Conversion (Colab Optimized)

This notebook handles batch conversion of PDF files to Markdown using `opendataloader-pdf`.

In [7]:
import sys
if 'google.colab' in sys.modules:
    # 1. Install system dependencies (Java JVM for Docling)
    print("Installing system dependencies (Java)...")
    !apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless
    
    # 2. Install Python package
    %pip install -q opendataloader-pdf[hybrid]
else:
    %pip install opendataloader-pdf[hybrid]

# 3. Start backend server
print("Starting backend server...")
!nohup opendataloader-pdf-hybrid --port 5002 --enrich-formula --enrich-picture-description > server.log 2>&1 &

# 4. Wait for server to be ready
import time, socket
for _ in range(30):
    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('localhost', 5002)) == 0:
                print("Server is ready!")
                break
    except:
        pass
    time.sleep(1)
else:
    print("Warning: Server status uncertain. Check server.log:")
    !tail -n 10 server.log

Installing system dependencies (Java)...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Starting backend server...
Server is ready!


## Step 2: Choose your File Source

In [8]:
import os
import sys
import glob

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)

    # 1. Define Drive paths
    # Local path 'G:\My Drive\temp' maps to '/content/drive/MyDrive/temp/' in Colab
    input_dir = "/content/drive/MyDrive/temp/"
    output_dir = "/content/drive/MyDrive/pdf_conversion_output/"
    
    os.makedirs(input_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    # 2. Automatically discover PDF files
    # This will find all .pdf files in your Drive's 'temp' folder
    valid_files = glob.glob(os.path.join(input_dir, "*.pdf"))

    if not valid_files:
        print(f"Warning: No PDF files found in {input_dir}")
        print("Tip: Place your PDFs in 'G:\\My Drive\\temp' and wait for them to sync.")
    else:
        print(f"Success: Found {len(valid_files)} PDF files in Drive.")
        for f in valid_files:
            print(f"  - {os.path.basename(f)}")
else:
    print("Note: Not running in Google Colab environment.")
    valid_files = []

Mounted at /content/drive
Success: Found 3 PDF files in Drive.
  - Assis et al. - 2025 - The performance-interpretability trade-off a comparative study of machine learning models.pdf
  - Badiola et al. - 2025 - Real-Time Subject-Specific Predictive Modeling of PPG Signals for Artifact-Resilient SpO2 Estimation.pdf
  - Chan et al. - 2021 - Enabling Continuous Wearable Reflectance Pulse Oximetry at the Sternum.pdf


In [9]:
## Step 3: Verify Files Ready
if 'valid_files' in locals() and valid_files:
    print(f"Success: {len(valid_files)} files ready for conversion.")
else:
    print("ERROR: No valid files identified. Please run Step 2.")

Success: 3 files ready for conversion.


In [10]:
## Step 4: Run Conversion
import opendataloader_pdf

if 'valid_files' in locals() and valid_files:
    print(f"Starting conversion to {output_dir}...")
    opendataloader_pdf.convert(
        input_path=valid_files,
        output_dir=output_dir,
        format="markdown",
        hybrid="docling-fast",
        hybrid_mode="full",
        hybrid_timeout="600000",
    )
    print("Conversion complete!")
else:
    print("Error: No valid files identified.")

Starting conversion to /content/drive/MyDrive/pdf_conversion_output/...
May 13, 2026 6:29:58 AM org.opendataloader.pdf.processors.DocumentProcessor preprocessing
INFO: File name: /content/drive/MyDrive/temp/Assis et al. - 2025 - The performance-interpretability trade-off a comparative study of machine learning models.pdf
May 13, 2026 6:30:01 AM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
May 13, 2026 6:30:01 AM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
May 13, 2026 6:30:01 AM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
May 13, 2026 6:30:01 AM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
May 13, 2026 6:30:01 AM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
May 13, 2026 6:30:01 AM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
May 13, 2026 6:30:01 AM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
May 13, 2026 6:30:01 AM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
May 1

In [11]:
## Step 5: Cleanup Input Files
import os
if 'valid_files' in locals() and valid_files:
    print("Cleaning up processed files...")
    for f in valid_files:
        try:
            os.remove(f)
            print(f"  - Deleted: {os.path.basename(f)}")
        except Exception as e:
            print(f"  - Error deleting {os.path.basename(f)}: {e}")
    print("Cleanup complete!")
else:
    print("No files to clean up.")

Cleaning up processed files...
  - Deleted: Assis et al. - 2025 - The performance-interpretability trade-off a comparative study of machine learning models.pdf
  - Deleted: Badiola et al. - 2025 - Real-Time Subject-Specific Predictive Modeling of PPG Signals for Artifact-Resilient SpO2 Estimation.pdf
  - Deleted: Chan et al. - 2021 - Enabling Continuous Wearable Reflectance Pulse Oximetry at the Sternum.pdf
Cleanup complete!


In [12]:
!pkill -f opendataloader-pdf-hybrid
print("Server closed.")

Server closed.
